# Database Integration with SQLModel



## Understanding the Stack: SQLAlchemy, SQLModel, and Pydantic

Before we set up our database, let's understand why we need three different tools:

### **SQLAlchemy** — The Database Layer
- Handles communication with the database (PostgreSQL, SQLite, etc.)
- Defines table structure as Python classes
- Generates SQL queries automatically
- Manages database sessions and transactions

### **Pydantic** — Data Validation & Serialization
- Validates incoming API request data (type-checking, constraints)
- Converts JSON into Python objects
- Serializes Python objects back to JSON for API responses
- Acts as the API contract between client and server

### **SQLModel** — Combining Both
- Unifies SQLAlchemy (database models) + Pydantic (validation schemas) into one class
- Reduces code duplication — define your model once, use it everywhere
- Works seamlessly with FastAPI

### The Data Flow
```
Client JSON → Pydantic validation → SQLModel instance → SQLAlchemy INSERT → Database
Database → SQLAlchemy query → SQLModel instance → Pydantic serialization → JSON response
```

Coming from Django/Flask, think of SQLModel as a model that does both database schema AND API validation in one place.

## Running the Server

Your FastAPI application is defined in `src/__init__.py` with a lifespan context manager that initializes the database on startup.

### Start the Server
```bash
uvicorn src:app --reload
```

**Important:** Use `src:app` (not `src.books.routes:app`) because the app is in `src/__init__.py`, not in the routes file.

### What Happens on Startup

When you start the server, the lifespan function runs and calls `initdb()`, which:
1. Connects to the database
2. Checks if the `books` table exists
3. Creates it if it doesn't (since `echo=True` is set)
4. Logs all SQL operations

### Expected Output

You should see SQLAlchemy debug output like:

```
INFO:     Waiting for application startup.
INFO sqlalchemy.engine.Engine BEGIN (implicit)
INFO sqlalchemy.engine.Engine PRAGMA table_info("books")
INFO sqlalchemy.engine.Engine CREATE TABLE books (
    id UUID PRIMARY KEY,
    title VARCHAR NOT NULL,
    ...
)
INFO sqlalchemy.engine.Engine COMMIT
INFO:     Application startup complete
```

This debug output confirms that:
- The database connection is working
- SQLModel is creating your database tables
- SQLAlchemy is logging all SQL queries (thanks to `echo=True` in the engine configuration)